# 31 · Capstone: one app, one agent, one Dataverse

## Goal

Close the loop: an app button fires `TriageOnDemand`, which calls the
agent, parses a structured recommendation out of its response, writes it
back to `crd_supplierrenewal`, and the app reflects the new value —
with the same security-role boundary from `26` proven to hold end to end.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from pathlib import Path
for f in ["SupplierBrowse.pa.yaml", "SupplierDetail.pa.yaml"]:
    assert (Path(f"../apps/renewal-desk-canvas/src/Screens/{f}")).exists(), "run 27-28 first"
assert (Path("../apps/renewal-desk-canvas/flows/TriageOnDemand.json")).exists(), "run 29-30 first"


## Concept

This is the round trip the whole elective track has been building toward,
and it's worth being precise about what "structured" means here: the
agent's response is still natural language (that's what `app-02-
structured-writeback` in the golden set checks for — a response *suitable*
for parsing, not JSON output, because `03`'s citation-suppression trap
already showed forcing rigid JSON output has real costs). The flow does a
light extraction pass on that text before the Dataverse write — a
deliberately small, auditable step, not a second LLM call for something
this constrained.

**Verification here isn't a chat eval — it's a Dataverse assertion.**
There's no `run_suite()` equivalent for "did the app's screen actually
update"; the closest honest proxy is asserting the Web API row changed,
and separately proving the security-role boundary from `26` still holds:
a `Procurement Analyst` token must be able to *read* the updated decision
but never *write* it back themselves.


## Build


### Wire the write-back (Power Automate designer)


On `TriageOnDemand`, after the HTTP call to the agent, add a "Parse
text" / simple extraction step for the recommendation (renew/escalate),
then a Dataverse "Update a row" action against `crd_supplierrenewal`'s
`crd_decision` field, keyed on the supplier passed in from the app.


In [ ]:
import subprocess
export = subprocess.run([
    "pac", "solution", "export", "--name", "crd-renewal-desk-flows",
    "--path", "../dist/renewal-desk-flows.zip", "--managed", "false",
], capture_output=True, text=True)
from csx.pac import solution_unpack
from pathlib import Path
solution_unpack(Path("../dist/renewal-desk-flows.zip"), Path("../apps/renewal-desk-canvas/flows/_unpacked"))
import shutil
for f in Path("../apps/renewal-desk-canvas/flows/_unpacked").rglob("*.json"):
    if "TriageOnDemand" in f.name:
        shutil.copy(f, Path("../apps/renewal-desk-canvas/flows") / f.name)


### Wire the app button (designer)

On `SupplierDetail`, add a button: `OnSelect: TriageOnDemand.Run(SupplierGallery.Selected.crd_SupplierName)`. Save — Git Integration syncs it.


## Verify

Same harness, same golden set, every notebook.


Trigger the round trip for real, then assert the Dataverse row actually changed — this is the capstone's real test, not a chat transcript.


In [ ]:
from csx.config import load_settings
from csx.clients import get_application_token
from csx.dataverse import get_row

settings = load_settings()
token = get_application_token(settings)

NORTHWIND_KEY = "crd_supplierrenewal_name='Northwind Fasteners'"
before = get_row(settings, token, "crd_supplierrenewals", NORTHWIND_KEY, select=["crd_decision"])
print("before:", before.get("crd_decision"))
print("Manually trigger TriageOnDemand for Northwind Fasteners from the app now, wait for the flow run to complete, then run the next cell.")


In [ ]:
after = get_row(settings, token, "crd_supplierrenewals", NORTHWIND_KEY, select=["crd_decision"])
print("crd_decision after round trip:", after.get("crd_decision"))
assert after.get("crd_decision"), "round trip did not write back a decision — check the flow run history for the failed step"


### Security-role boundary, proven end to end


In [ ]:
from csx.clients import get_delegated_token  # sign in as the Procurement Analyst test user when prompted
from csx.dataverse import assert_security_role_scoped

analyst_token = get_delegated_token(settings)  # Procurement Analyst — read-only per 26
denied_write = assert_security_role_scoped(settings, analyst_token, "crd_supplierrenewals", "<row-id>")
print(f"Procurement Analyst correctly denied write? {denied_write}")
assert denied_write, "a read-only role could write to the decision field — the security role from 26 isn't actually scoping the app"


## Cost


In [ ]:
from csx.cost import CreditMeter
meter = CreditMeter(environment_id=settings.get("DATAVERSE_ENV_ID"))
meter.report_cost("31", budget=settings.get("COPILOT_CREDIT_BUDGET"), delta_credits=1,
                   note="one round-trip invocation; Power Apps/Automate licensing (per-app vs per-user, premium Dataverse connector) is separate from Copilot Credits — budget both before a production rollout")


## Teardown


In [ ]:
print("End of Part 2's build arc. T10-bonus wires this app + its flows into the same CI/CD pipeline T9-bonus built for the agent fleet.")
